In [1]:
# =========================
# 	 LOAD & ADD DOCUMENT
# =========================

import sys

sys.path.append("../")
from document_loader import document_loader
from vector_db import vector_db

collection_name = ["fixed-chunk","recursive-chunk", "semantic-chunk"]

for name in collection_name:
  documents = document_loader(name)
  vector_db.add_document(
    collection_name = name,
    documents = documents
  )

Berhasil menambahkan 432 dokumen ke koleksi 'fixed-chunk'
Berhasil menambahkan 460 dokumen ke koleksi 'recursive-chunk'
Berhasil menambahkan 96 dokumen ke koleksi 'semantic-chunk'


In [4]:
# ===============
# 	GET CONTEXT
# ===============

import sys
import pandas as pd
import time

sys.path.append("../")
from vector_db import vector_db

collection_name = "semantic-chunk"

data_test = pd.read_csv("../document/data_test_rag1.csv", sep = ";")
retriever = vector_db.get_retriever(collection_name)

def get_context(question):
  t0 = time.perf_counter()
  docs = retriever.invoke(question)
  total_time = round(time.perf_counter() - t0, 2)

  context = "\n\n".join([doc.page_content for doc in docs])
  chunk_id = [doc.metadata.get("chunk_id") for doc in docs]

  return context, chunk_id, total_time

retrieved_context = []
chunk_id = []
duration = []

for question in data_test["question"]:
  context, id, t = get_context(question)
  
  retrieved_context.append(context)
  chunk_id.append(id)
  duration.append(t)

context_result = pd.DataFrame({
  f"{collection_name}_retrieved_context": retrieved_context,
  f"{collection_name}_id": chunk_id,
  f"{collection_name}_duration": duration
})

try:
  df = pd.read_csv(f"../result_data/result_data.csv", sep=";")
  df = pd.concat([df, context_result], axis=1)
except FileNotFoundError:
  df = context_result

df.to_csv(f"../result_data/result_data.csv", sep=";", index=False)

In [7]:
# ===============
# 	GET ANSWER
# ===============

import sys
import pandas as pd

sys.path.append("../")
from rag import rag

collection_name = "semantic-chunk"

data_test = pd.read_csv("../document/data_test_rag1.csv", sep = ";")
result_context = pd.read_csv(f"../result_data/result_data.csv", sep = ";")

questions = data_test["question"].tolist()
contexts = result_context[f"{collection_name}_retrieved_context"].to_list()

llm_response = []
duration_response = []

for question, context in zip(questions, contexts):
  response, t = rag.get_answer(question, context)

  llm_response.append(response)
  duration_response.append(t)

rag_result = pd.DataFrame({
  f"{collection_name}_llm_response": llm_response,
  f"{collection_name}_duration_response": duration_response
})

try:
  df = pd.read_csv("../result_data/result_data.csv", sep=";")
  df = pd.concat([df, rag_result], axis=1)

except FileNotFoundError:
  df = rag_result

df.to_csv("../result_data/result_data.csv", sep=";", index=False)

In [3]:
# =========================
# 	RETRIEVER EVALUATION
# =========================

import pandas as pd
import ast

collection_name = "semantic-chunk"
df_result = pd.read_csv("../result_data/result_data.csv", sep=";")
df_test = pd.read_csv("../document/data_test_rag1.csv", sep=";")

column = f"{collection_name}_id"
retrieved_chunk = df_result[column].apply(ast.literal_eval)
reference_chunk = df_test[column].apply(ast.literal_eval)

def retriever_eval(retrieved, reference):
  precision = len(set(retrieved) & set(reference)) / len(retrieved)
  recall = len(set(retrieved) & set(reference)) / len(reference)
  f1 = 2 * precision * recall / (precision + recall + 1e-10)

  return precision, recall, f1

result = []

for ret, ref in zip(retrieved_chunk, reference_chunk):
  scores = {} 
  precision, recall, f1 = retriever_eval(ret, ref)

  scores[f"{collection_name}_precision"] = precision
  scores[f"{collection_name}_recall"] = recall
  scores[f"{collection_name}_f1"] = f1

  result.append(scores)

df_eval_result = pd.DataFrame(result)

try:
  df = pd.read_csv("../result_data/eval_result.csv", sep=";")
  df = pd.concat([df, df_eval_result], axis=1)

except FileNotFoundError:
  df = df_eval_result

df.to_csv("../result_data/eval_result.csv", sep=";", index=False)

In [ ]:
# ====================
#   RAGAS EVALUATION
# ====================

import sys
import types

dummy_chat = types.ModuleType("langchain_community.chat_models.vertexai")
dummy_chat.ChatVertexAI = type("ChatVertexAI", (object,), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat

import langchain_community.llms
langchain_community.llms.VertexAI = type("VertexAI", (object,), {})

In [16]:
from datasets import Dataset
import pandas as pd

collection_name = "fixed-chunk"

df_test = pd.read_csv("../document/data_test_rag1.csv", sep = ";")
df_result = pd.read_csv("../result_data/result_data.csv", sep = ";")

df_eval = pd.concat([df_test, df_result], axis=1)
df_eval = df_eval[
  [
    "question",
    f"{collection_name}_llm_response",
  ]
].copy()

df_eval.rename(
  columns={
    "question": "user_input",
    f"{collection_name}_llm_response": "response",
  },
  inplace=True,
)

dataset = Dataset.from_pandas(
  df_eval,
  preserve_index=False
)
dataset

Dataset({
    features: ['user_input', 'response'],
    num_rows: 117
})

In [17]:
from ragas import evaluate
from ragas.run_config import RunConfig
from langchain_ollama import ChatOllama
from ragas.llms import LangchainLLMWrapper
from langchain_ollama import OllamaEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
  answer_relevancy,
  context_precision,
  context_recall,
)

llm = ChatOllama( model="qwen3:8b", temperature=0 ) 
embedding = OllamaEmbeddings( model="bge-m3" )

evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embedding = LangchainEmbeddingsWrapper(embedding)

run_config = RunConfig(
  timeout=600,
  max_workers=1,
  max_retries=3,
  max_wait=60,
)

result = evaluate(
  dataset=dataset,
  metrics=[
    answer_relevancy
  ],
  llm=evaluator_llm,
  embeddings=evaluator_embedding,
  run_config=run_config
)

C:\Users\labma\AppData\Local\Temp\ipykernel_9528\4124614949.py:7: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\labma\AppData\Local\Temp\ipykernel_9528\4124614949.py:7: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\labma\AppData\Local\Temp\ipykernel_9528\4124614949.py:7: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
C:\Users\labma\AppData\Local\Temp\ipykernel_9

In [18]:
df_eval_result = result.to_pandas()
df_eval_result.rename(
  columns= {
    "user_input": "question",
    "response": f"{collection_name}_llm_response",
    "answer_relevancy": f"{collection_name}_answer_relevancy"
	},
  inplace= True
)

try:
  df = pd.read_csv("../result_data/eval_result.csv", sep=";")
  df = pd.concat([df, df_eval_result], axis=1)

except FileNotFoundError:
  df = df_eval_result

df.to_csv("../result_data/eval_result.csv", sep=";", index=False)